In [20]:
# ── Cell 1: Install & Imports ────────────────────────────────
import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [21]:
# ── Cell 2: Configuration ────────────────────────────────────

class Config:
    # Paths — updated to match actual extracted structure
    EXTRACT_DIR        = "/content/real_vs_fake/real-vs-fake"
    SAVE_DIR           = "/content/drive/MyDrive/TruthLens AI Model/"
    MODEL_SAVE_PATH    = os.path.join(SAVE_DIR, "deepfake_detector.keras")
    CSV_LOG_PATH       = os.path.join(SAVE_DIR, "training_log.csv")

    IMG_SIZE           = 224
    CHANNELS           = 3

    BATCH_SIZE         = 32
    PHASE1_EPOCHS      = 15
    PHASE2_EPOCHS      = 25
    PHASE1_LR          = 1e-3
    PHASE2_LR          = 1e-5

    DROPOUT_RATE       = 0.4
    L2_LAMBDA          = 1e-4
    UNFREEZE_LAYERS    = 30

    # Updated: folder names are 'real' and 'fake' (not '0' and '1')
    # class_mode='binary' with classes=['real','fake'] means:
    #   'real' → index 0 (REAL), 'fake' → index 1 (FAKE)
    CLASSES            = ['real', 'fake']

cfg = Config()

In [22]:
# ── Cell 3: Mount Google Drive & Extract Dataset ─────────────

def mount_and_extract():
    """Mount Google Drive and extract the dataset ZIP if not already extracted."""
    from google.colab import drive
    drive.mount("/content/drive")
    print("✅ Google Drive mounted")

    if not os.path.exists(cfg.EXTRACT_DIR):
        print("📦 Extracting dataset...")
        with zipfile.ZipFile(cfg.DRIVE_DATASET_ZIP, 'r') as zip_ref:
            zip_ref.extractall("/content/")
        print("✅ Dataset extracted")
    else:
        print("✅ Dataset already extracted, skipping")

    # Print class counts for sanity check
    for split in ['train', 'validate', 'test']:
        for label in cfg.CLASSES:
            path = os.path.join(cfg.EXTRACT_DIR, split, label)
            if os.path.exists(path):
                count = len(os.listdir(path))
                label_name = "REAL" if label == '0' else "FAKE"
                print(f"  {split}/{label} ({label_name}): {count} images")

mount_and_extract()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
✅ Dataset already extracted, skipping
  train/real (FAKE): 50000 images
  train/fake (FAKE): 50000 images
  test/real (FAKE): 10000 images
  test/fake (FAKE): 10000 images


In [23]:
# ── Cell 4: Data Generators ──────────────────────────────────

def build_data_generators():
    train_datagen = ImageDataGenerator(
        horizontal_flip      = True,
        rotation_range       = 15,
        brightness_range     = [0.7, 1.3],
        zoom_range           = 0.15,
        width_shift_range    = 0.1,
        height_shift_range   = 0.1,
        shear_range          = 0.1,
        fill_mode            = 'nearest'
    )

    eval_datagen = ImageDataGenerator()

    train_gen = train_datagen.flow_from_directory(
        directory   = os.path.join(cfg.EXTRACT_DIR, 'train'),
        target_size = (cfg.IMG_SIZE, cfg.IMG_SIZE),
        batch_size  = cfg.BATCH_SIZE,
        class_mode  = 'binary',
        classes     = cfg.CLASSES,   # 'real'→0, 'fake'→1
        shuffle     = True,
        seed        = 42
    )

    val_gen = eval_datagen.flow_from_directory(
        directory   = os.path.join(cfg.EXTRACT_DIR, 'valid'),  # 'valid' not 'validate'
        target_size = (cfg.IMG_SIZE, cfg.IMG_SIZE),
        batch_size  = cfg.BATCH_SIZE,
        class_mode  = 'binary',
        classes     = cfg.CLASSES,
        shuffle     = False
    )

    test_gen = eval_datagen.flow_from_directory(
        directory   = os.path.join(cfg.EXTRACT_DIR, 'test'),
        target_size = (cfg.IMG_SIZE, cfg.IMG_SIZE),
        batch_size  = cfg.BATCH_SIZE,
        class_mode  = 'binary',
        classes     = cfg.CLASSES,
        shuffle     = False
    )

    print(f"📊 Class indices: {train_gen.class_indices}")
    print(f"   ↳ 'real' must map to 0, 'fake' must map to 1")
    assert train_gen.class_indices == {'real': 0, 'fake': 1}, \
        "❌ Label mapping wrong!"

    print(f"\n   Train images:      {train_gen.samples:,}")
    print(f"   Validation images: {val_gen.samples:,}")
    print(f"   Test images:       {test_gen.samples:,}")

    return train_gen, val_gen, test_gen


train_gen, val_gen, test_gen = build_data_generators()

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
📊 Class indices: {'real': 0, 'fake': 1}
   ↳ 'real' must map to 0, 'fake' must map to 1

   Train images:      100,000
   Validation images: 20,000
   Test images:       20,000


In [24]:
# ── Cell 5: Compute Class Weights ────────────────────────────

# --- Debug: print exactly what the generator sees ──────────
print("class_indices:", train_gen.class_indices)
print("unique classes in labels:", np.unique(train_gen.classes))
print("total labels:", len(train_gen.classes))

def compute_class_weights(train_gen):
    """
    Compute class weights using only what the generator actually contains.
    Derives class indices directly from the generator instead of hardcoding.
    """
    labels      = train_gen.classes
    total       = len(labels)
    num_classes = len(train_gen.class_indices)

    class_weight_dict = {}
    for class_name, class_idx in train_gen.class_indices.items():
        count = np.sum(labels == class_idx)
        weight = total / (num_classes * count) if count > 0 else 1.0
        class_weight_dict[class_idx] = weight
        label_name = "REAL" if class_name == '0' else "FAKE"
        print(f"   {label_name} ({class_name} → idx {class_idx}): {count:,} images → weight: {weight:.4f}")

    print(f"\n⚖️  Class weights: {class_weight_dict}")
    return class_weight_dict


class_weights = compute_class_weights(train_gen)

class_indices: {'real': 0, 'fake': 1}
unique classes in labels: [0 1]
total labels: 100000
   FAKE (real → idx 0): 50,000 images → weight: 1.0000
   FAKE (fake → idx 1): 50,000 images → weight: 1.0000

⚖️  Class weights: {0: np.float64(1.0), 1: np.float64(1.0)}


In [25]:
# ── Cell 6: Build Model ──────────────────────────────────────

def build_model():
    """
    Build EfficientNetB0-based deepfake detector.

    Architecture:
      EfficientNetB0 (pretrained ImageNet, frozen)
        → GlobalAveragePooling2D
        → BatchNorm + Dense(256) + Dropout
        → BatchNorm + Dense(64) + Dropout
        → Dense(1, sigmoid)   ← binary output: 0=REAL, 1=FAKE

    Output shape: (None, 1)  — compatible with detector.py
    Threshold:    score > 0.5 → FAKE  (as expected by backend)
    """

    # --- Base model: EfficientNetB0 pretrained on ImageNet ─────
    # include_top=False removes the original classification head
    # input_shape uses 3 channels (RGB)
    base_model = EfficientNetB0(
        weights      = 'imagenet',
        include_top  = False,
        input_shape  = (cfg.IMG_SIZE, cfg.IMG_SIZE, cfg.CHANNELS)
    )

    # Freeze entire base for Phase 1 (we only train the new head)
    base_model.trainable = False
    print(f"🔒 Base model layers: {len(base_model.layers)} (all frozen for Phase 1)")

    # --- Custom classification head ────────────────────────────
    inputs = keras.Input(shape=(cfg.IMG_SIZE, cfg.IMG_SIZE, cfg.CHANNELS))

    # Pass through EfficientNetB0 (no training here in Phase 1)
    x = base_model(inputs, training=False)

    # Pooling — converts feature maps (7x7xC) to a 1D vector
    x = layers.GlobalAveragePooling2D()(x)

    # --- First dense block ─────────────────────────────────────
    x = layers.BatchNormalization()(x)
    x = layers.Dense(
        256,
        activation    = 'relu',
        kernel_regularizer = regularizers.l2(cfg.L2_LAMBDA)  # Prevent overfitting
    )(x)
    x = layers.Dropout(cfg.DROPOUT_RATE)(x)

    # --- Second dense block ────────────────────────────────────
    x = layers.BatchNormalization()(x)
    x = layers.Dense(
        64,
        activation    = 'relu',
        kernel_regularizer = regularizers.l2(cfg.L2_LAMBDA)
    )(x)
    x = layers.Dropout(cfg.DROPOUT_RATE / 2)(x)  # Less dropout near output

    # --- Output layer ──────────────────────────────────────────
    # Sigmoid outputs a probability in [0, 1]
    # 0 → REAL, 1 → FAKE (matches backend threshold: score > 0.5 = FAKE)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    return model, base_model


model, base_model = build_model()
model.summary()

🔒 Base model layers: 238 (all frozen for Phase 1)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,400,164 (16.79 MB)

 Trainable params: 347,521 (1.33 MB)

 Non-trainable params: 4,052,643 (15.46 MB)

In [26]:
# ── Cell 7: Callbacks ────────────────────────────────────────

def build_callbacks(phase: int):
    """
    Build training callbacks.
    - ModelCheckpoint: saves the best model by val_accuracy
    - EarlyStopping:   stops if val_loss doesn't improve (patience varies by phase)
    - ReduceLROnPlateau: cuts LR when val_loss plateaus
    - CSVLogger: saves all epoch metrics to CSV on Drive
    """

    # Temporary checkpoint path (will be saved to Drive at end)
    checkpoint_path = f"/content/best_model_phase{phase}.keras"

    callbacks = [
        # Save best model during training
        ModelCheckpoint(
            filepath          = checkpoint_path,
            monitor           = 'val_accuracy',
            save_best_only    = True,
            save_weights_only = False,
            mode              = 'max',
            verbose           = 1
        ),

        # Stop early if val_loss stops improving
        EarlyStopping(
            monitor              = 'val_loss',
            patience             = 6 if phase == 1 else 8,  # More patience in fine-tune
            restore_best_weights = True,
            verbose              = 1
        ),

        # Reduce learning rate when stuck
        ReduceLROnPlateau(
            monitor  = 'val_loss',
            factor   = 0.3,       # Multiply LR by 0.3 on plateau
            patience = 3,
            min_lr   = 1e-7,
            verbose  = 1
        ),

        # Log metrics to CSV for post-analysis
        CSVLogger(cfg.CSV_LOG_PATH, append=True)
    ]

    return callbacks, checkpoint_path

In [ ]:
# ── Cell 8: Phase 1 Training — Frozen Base ───────────────────

def train_phase1():
    """
    Phase 1: Train only the custom head with the EfficientNetB0 base frozen.
    This lets the new Dense layers learn quickly without disturbing
    the ImageNet features already learned by the base.
    """
    print("\n" + "="*60)
    print("🚀 PHASE 1: Training custom head (base frozen)")
    print("="*60)

    # Compile with higher learning rate — fresh layers can handle it
    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=cfg.PHASE1_LR),
        loss      = 'binary_crossentropy',
        metrics   = ['accuracy', keras.metrics.AUC(name='auc')]
    )

    callbacks, checkpoint_path = build_callbacks(phase=1)

    history1 = model.fit(
        train_gen,
        epochs           = cfg.PHASE1_EPOCHS,
        validation_data  = val_gen,
        class_weight     = class_weights,   # Handle imbalance
        callbacks        = callbacks,
        verbose          = 1
    )

    print(f"\n✅ Phase 1 complete. Best model saved at: {checkpoint_path}")
    return history1, checkpoint_path


history1, phase1_checkpoint = train_phase1()


🚀 PHASE 1: Training custom head (base frozen)
Epoch 1/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step - accuracy: 0.7376 - auc: 0.8142 - loss: 0.5929
Epoch 1: val_accuracy improved from None to 0.84380, saving model to /content/best_model_phase1.keras

Epoch 1: finished saving model to /content/best_model_phase1.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 1579s 495ms/step - accuracy: 0.7699 - auc: 0.8499 - loss: 0.5449 - val_accuracy: 0.8438 - val_auc: 0.9246 - val_loss: 0.4315 - learning_rate: 0.0010
Epoch 2/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 458ms/step - accuracy: 0.8032 - auc: 0.8832 - loss: 0.4965
Epoch 2: val_accuracy improved from 0.84380 to 0.86595, saving model to /content/best_model_phase1.keras

Epoch 2: finished saving model to /content/best_model_phase1.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 1466s 469ms/step - accuracy: 0.8041 - auc: 0.8857 - loss: 0.4943 - val_accuracy: 0.8659 - val_auc: 0.9390 - val_loss: 0.4010 - learning_rate: 0.0010
Epoch 3/15
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 